# AI Gym Trainer - Real Transformer Model
Full training pipeline

In [ ]:
!pip install opencv-python mediapipe torch torchvision scikit-learn tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_PATH = "/content/drive/MyDrive/gym_dataset"

In [ ]:
import cv2, mediapipe as mp, numpy as np
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

def extract_keypoints(video_path):
    cap = cv2.VideoCapture(video_path)
    keypoints = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        if results.pose_landmarks:
            frame_kp = []
            for lm in results.pose_landmarks.landmark:
                frame_kp.extend([lm.x, lm.y, lm.z])
            keypoints.append(frame_kp)
    cap.release()
    return keypoints

In [ ]:
import os
from tqdm import tqdm
X, y, label_map = [], [], {}
label_id = 0
for exercise in os.listdir(DATASET_PATH):
    path = os.path.join(DATASET_PATH, exercise)
    if not os.path.isdir(path): continue
    label_map[label_id] = exercise
    for video in tqdm(os.listdir(path)):
        kp = extract_keypoints(os.path.join(path, video))
        if len(kp) > 10:
            X.append(kp)
            y.append(label_id)
    label_id += 1
print(label_map)

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
X = [torch.tensor(seq, dtype=torch.float32) for seq in X]
X_padded = pad_sequence(X, batch_first=True)
y = torch.tensor(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X_padded, y, test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

In [ ]:
import torch.nn as nn
class TransformerModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.embedding = nn.Linear(input_dim, 128)
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerModel(99, len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for i in range(len(X_train)):
        x = X_train[i].unsqueeze(0).to(device)
        target = y_train[i].unsqueeze(0).to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss}")

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for i in range(len(X_test)):
        x = X_test[i].unsqueeze(0).to(device)
        target = y_test[i].to(device)
        pred = model(x).argmax()
        if pred == target:
            correct += 1
print("Test Accuracy:", correct/len(X_test))